In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#For converting low/medium/high into 0/1/2 and Feature_4 into values
from sklearn.preprocessing import OrdinalEncoder
#Target Encoding
from sklearn.preprocessing import TargetEncoder
#Importing XGBoost Model
from xgboost import XGBRegressor
#Importing LightGBM Model
import lightgbm as lgb
#Train Test Split
from sklearn.model_selection import train_test_split
#Model Evaluation 
from sklearn.metrics import mean_squared_error, r2_score
#Importing K-Fold Cross Validation
from sklearn.model_selection import KFold, cross_val_score

In [2]:
#Reading the training data
df = pd.read_csv('train-2.csv')

In [3]:
# Explicit Mapping For Features 1,2,3
ordinal_map = {'Low': 0, 'Medium': 1, 'High': 2}
for col in ['Feature_1', 'Feature_2', 'Feature_3']:
    df[col] = df[col].map(ordinal_map)

In [5]:
#Frequency Encoding Feature 4
freq_encoding = df['Feature_4'].value_counts()
df['Feature_4'] = df['Feature_4'].map(freq_encoding)

In [6]:
#Performing Pearson Correlation Analysis and removing redundant columns
num_cols = [c for c in df.select_dtypes(include='number').columns 
            if c not in ['id', 'target']]

corr_matrix = df[num_cols].corr()


pairs = []
for i, col1 in enumerate(num_cols):
    for j, col2 in enumerate(num_cols):
        if i < j:                              
            val = corr_matrix.loc[col1, col2]
            if abs(val) >= 0.98:                 
                pairs.append([col1, col2])
df = df.drop(columns = ['Feature_7','Feature_9','Feature_22','Feature_15','Feature_18','Feature_27','Feature_47'])

In [7]:
#Performing Median Imputation for filling Null values
df = df.fillna(df.median(numeric_only=True))

In [8]:
#Seperating the training data into features and labels
X = df.drop(columns=['id','target'])
Y = df['target']

In [9]:
#Creating 5 Fold Cross Validation 
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [10]:
#Creating Train-Test split and building XGBoost Model
X_train, X_test, Y_train, Y_test = train_test_split(X,Y,test_size=0.2,random_state=42)
Y_train_log = np.log(Y_train)

X_for_cross_val = X.copy()

xgb_model = XGBRegressor(
    n_estimators=1000,       
    max_depth=6,            
    learning_rate=0.1,      
    objective='reg:squarederror', 
    random_state=42)
xgb_model.fit(X_train,Y_train_log)
predictions = np.exp(xgb_model.predict(X_test))
mse = mean_squared_error(Y_test, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(Y_test, predictions)
scores_xgb = cross_val_score(xgb_model, X_for_cross_val, np.log(Y), cv=kf, scoring='r2')
print(f"Mean Accuracy: {np.mean(scores_xgb):.4f}")
print(r2)

Mean Accuracy: 0.8953
0.8249334927992176


In [11]:
#Building LGBoost Model
lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.8,
    random_state=42
)
lgb_model.fit(X_train,Y_train_log)
prediction = np.exp(lgb_model.predict(X_test))
r2s = r2_score(Y_test, prediction)
scores_lgb = cross_val_score(lgb_model, X_for_cross_val, np.log(Y), cv=kf, scoring='r2')
print(f"Mean Accuracy: {np.mean(scores_lgb):.4f}")
print(r2s)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007375 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12160
[LightGBM] [Info] Number of data points in the train set: 11907, number of used features: 54
[LightGBM] [Info] Start training from score 0.915147
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005210 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12160
[LightGBM] [Info] Number of data points in the train set: 11907, number of used features: 54
[LightGBM] [Info] Start training from score 0.915147
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004881 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12151
[LightGBM] [Info] Number of data points in the train set: 11907, number of used features: 54
[LightGBM] [Info] Start 

In [12]:
#Building CatBoost Model 
import catboost as cb

cb_model = cb.CatBoostRegressor(
    iterations=1000,
    learning_rate=0.03,
    depth=6,
    random_seed=42,
    verbose=0
)
cb_model.fit(X_train, Y_train_log)
cb_pred = np.exp(cb_model.predict(X_test))
cb_r2   = r2_score(Y_test, cb_pred)
scores_cb = cross_val_score(cb_model, X, np.log(Y), cv=kf, scoring='r2')
print(f"Mean Accuracy: {np.mean(scores_cb):.4f}")
print(cb_r2)

Mean Accuracy: 0.8781
0.8002821505272395


In [13]:
# Create arrays to hold the OOF predictions for the entire training dataset
oof_xgb = np.zeros(len(X))
oof_lgb = np.zeros(len(X))
oof_cb = np.zeros(len(X))

for train_index, val_index in kf.split(X):
    X_tr, X_val = X.iloc[train_index], X.iloc[val_index]
    Y_tr, Y_val = Y.iloc[train_index], Y.iloc[val_index]
    
    Y_tr_log = np.log(Y_tr)

    xgb_model.fit(X_tr, Y_tr_log)
    lgb_model.fit(X_tr, Y_tr_log)
    cb_model.fit(X_tr, Y_tr_log)
    
    oof_xgb[val_index] = np.exp(xgb_model.predict(X_val))
    oof_lgb[val_index] = np.exp(lgb_model.predict(X_val))
    oof_cb[val_index]  = np.exp(cb_model.predict(X_val))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007819 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12160
[LightGBM] [Info] Number of data points in the train set: 11907, number of used features: 54
[LightGBM] [Info] Start training from score 0.915147
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007913 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12151
[LightGBM] [Info] Number of data points in the train set: 11907, number of used features: 54
[LightGBM] [Info] Start training from score 0.923864
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008123 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12154
[LightGBM] [Info] Number of data points in the train set: 11907, number of used features: 54
[LightGBM] [Info] Start 

In [14]:
#Building Ensemble Model using Nelder_Mead Method to assign weights
import numpy as np
from sklearn.metrics import r2_score
from scipy.optimize import minimize

# Step 2 — find optimal weights using Nelder-Mead
def blend_loss(weights):
    w = np.abs(weights) / np.abs(weights).sum()
    blended = w[0]*oof_xgb + w[1]*oof_lgb + w[2]*oof_cb
    return -r2_score(Y, blended)

result = minimize(blend_loss, [1/3, 1/3, 1/3], method='Nelder-Mead')
w = np.abs(result.x) / np.abs(result.x).sum()

print('Robust XGBoost weight: ', round(w[0], 3))
print('Robust LightGBM weight:', round(w[1], 3))
print('Robust CatBoost weight:', round(w[2], 3))

Robust XGBoost weight:  0.316
Robust LightGBM weight: 0.458
Robust CatBoost weight: 0.225


In [15]:
#Performing EDA and Feature Engineering on test data
test_df = pd.read_csv('test.csv')

test_ids = test_df['id']

for col in ['Feature_1', 'Feature_2', 'Feature_3']:
    test_df[col] = test_df[col].map(ordinal_map)

test_df = test_df.fillna(test_df.median(numeric_only=True))

test_df['Feature_4'] = test_df['Feature_4'].map(freq_encoding).fillna(1)

test_df = test_df.drop(columns = ['Feature_7','Feature_9','Feature_22','Feature_15','Feature_18','Feature_27','Feature_47'])

X_test_submission = test_df.drop(columns=['id'])

In [16]:
# 1. Prepare the full target variable
Y_log = np.log(Y)

# 2. Retraining the models on the entire dataset
print("Retraining models on full dataset...")
xgb_model.fit(X, Y_log)
lgb_model.fit(X, Y_log)
cb_model.fit(X, Y_log)

# 3. Generate predictions on the final submission test set
print("Generating final predictions...")
xgb_test_pred_log = xgb_model.predict(X_test_submission)
lgb_test_pred_log = lgb_model.predict(X_test_submission)
cb_test_pred_log  = cb_model.predict(X_test_submission)

# 4. Exponentiate predictions to return them to the original scale
xgb_test_pred = np.exp(xgb_test_pred_log)
lgb_test_pred = np.exp(lgb_test_pred_log)
cb_test_pred  = np.exp(cb_test_pred_log)

# 5. Blend the predictions using robust OOF weights
# w[0] = XGB, w[1] = LGB, w[2] = CB
final_test_pred = (w[0] * xgb_test_pred) + (w[1] * lgb_test_pred) + (w[2] * cb_test_pred)

# 6. Create and save the submission file
submission_df = pd.DataFrame({
    'id': test_ids,
    'target': final_test_pred
})

submission_df.to_csv('submission.csv', index=False)
print("Submission saved successfully!")
display(submission_df.head())

Retraining models on full dataset...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002696 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 12159
[LightGBM] [Info] Number of data points in the train set: 14884, number of used features: 54
[LightGBM] [Info] Start training from score 0.920237
Generating final predictions...
Submission saved successfully!


,id,target
0,14884,1.108844
1,14885,8.772828
2,14886,10.974792
3,14887,1.129543
4,14888,10.695867
